# J2 — Les données avant les modèles

**Séance 2.** Un modèle ne vaut pas mieux que ses données. Aujourd'hui on regarde les données,
et on interroge la cible.

> **La règle de la séance : un constat sans chiffre n'est pas un constat.**
> « Il y a des valeurs manquantes » ne dit rien. « 5,4 % de manquants sur `solde_credit`, et les
> manquants ne sont pas corrélés à la cible » est un constat : il peut être vérifié, et il dit
> quoi faire.

**Livrable** : un rapport de **5 constats chiffrés** + **3 hypothèses de features**. Les gabarits
sont en fin de notebook — ce sont eux qu'on remplit, pas une page blanche.

In [ ]:
# 0 — les bibliothèques du cours, et les versions pour lesquelles il a été écrit :
#        pandas 3.0.6 · numpy 2.4.3 · scikit-learn 1.9.0 · jinja2
#
# Le code tourne AUSSI sous pandas 2 (c'est ce que Kaggle fournit) : les deux sont testés.
#
# SUR KAGGLE : rien à faire. Les bibliothèques sont déjà installées et l'INTERNET EST COUPÉ :
# une commande d'installation y échoue. Ce n'est pas une erreur de votre part.
#
# EN LOCAL, une seule fois — dans un terminal :
#     pip install "pandas==3.0.6" "numpy==2.4.3" "scikit-learn==1.9.0" "jinja2"
# dans une cellule d'un notebook local, la même chose avec un point d'exclamation devant.

# Ce que vous avez, à cet instant :
import importlib

for nom in ("pandas", "numpy", "sklearn", "jinja2"):
    try:
        print(f"{nom:<8}", importlib.import_module(nom).__version__)
    except ImportError:
        print(f"{nom:<8} absent — voir les lignes d'installation ci-dessus")

In [ ]:
# 1 — les outils
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
pd.set_option("display.width", 160)

In [ ]:
# 2 — où sont les fichiers (le notebook marche sur Kaggle ET en local)
# On CHERCHE train.csv au lieu de supposer un chemin : Kaggle a déjà déplacé ses points de
# montage, et un chemin recopié à la main se périme.
def _trouver_dossier(racines):
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie", os.path.join("..", "sortie"),
                            os.path.join("..", "Donnees_competition", "sortie")])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)

In [ ]:
# 3 — l'inventaire : une ligne par colonne. C'est la feuille de travail de la séance
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

inventaire = pd.DataFrame({
    "type_pandas": [str(t) for t in train.dtypes],
    "non_nuls": train.notna().sum().values,
    "manquants_%": (100 * train.isna().mean()).round(2).values,
    "valeurs_distinctes": [train[c].nunique(dropna=True) for c in train.columns],
    "exemple": [train[c].dropna().iloc[0] if train[c].notna().any() else "" for c in train.columns],
}, index=train.columns)

print("l'inventaire, trié par nombre de valeurs distinctes \n")
inventaire.sort_values("valeurs_distinctes").style
inventaire

In [ ]:
# 4 — les doublons, et les collisions d'identifiants
print("les doublons \n")
print("lignes dupliquées (toutes colonnes confondues) :", int(train.duplicated().sum()))
print("identifiants dupliqués :", int(train[ID].duplicated().sum()))

for col in ("dossier_id", "patient_id"):
    if col in train.columns:
        print(f"  {col:<14} distincts : {train[col].nunique():>7}  ·  lignes : {len(train):>7}")

# Un patient peut revenir plusieurs fois. Combien, au plus ?
if "patient_id" in train.columns:
    print("rendez-vous par patient — max :", int(train["patient_id"].value_counts().max()),
          "· médiane :", int(train["patient_id"].value_counts().median()))

In [ ]:
# 5 — la distribution de la cible, et la référence
# Sans la référence, aucun score ne veut rien dire — on la garde pour tout le semestre.
y = train[CIBLE]

print("la cible \n")
print(f"cible : {100*y.mean():.2f} % de positifs   ({int(y.sum())} sur {len(y)})")
print("valeurs présentes :", sorted(y.unique().tolist()))

reference = 100 * max(y.mean(), 1 - y.mean())

print("la référence, pour tout le semestre \n")
print(f"référence (classe majoritaire) : {reference:.2f} %")

## La cible est un objet suspect

Une cible n'est pas une vérité tombée du ciel : **quelqu'un l'a produite**, à un moment donné,
avec un protocole donné. Trois questions, et leurs réponses se mesurent :

1. **Qui** a produit ces étiquettes ? (un humain, un système, une règle ?)
2. **Quand** — avant ou après que le patient se présente ?
3. **Avec quel accord** entre annotateurs ? Une cible à 70 % d'accord est une cible **molle** :
   cela change tout le projet, y compris le score maximum atteignable.

On répond à la troisième par la mesure, pas par l'impression.

In [ ]:
# 6 — la matrice de travail : colonnes gardées, catégorielles, écartées
def matrice(cols):
    """Renvoie (X, colonnes catégorielles, colonnes écartées) pour un tableau donné."""
    A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
                 "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance", CIBLE]
    X = cols.drop(columns=[c for c in A_ECARTER if c in cols.columns]).copy()

    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]

    for c in cat:
        X[c] = X[c].fillna("").astype("category")

    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    # Xt est déjà catégorielle (matrice() l'a fait) : on ne fait qu'aligner.
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("les colonnes de travail \n")
print("colonnes utilisées :", X.shape[1])
print("écartées (cardinalité > 255) :", TROP_FIN)

In [ ]:
# 7 — le plafond de score selon la qualité de la cible
# On injecte du bruit : une proportion des étiquettes est retournée au hasard.
# Ce que ça montre : le plafond dépend de la QUALITÉ de la cible, avant tout modèle.
Xt_, Xv_, yt_, yv_ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

resultats = []
for bruit in (0.0, 0.05, 0.15, 0.30):
    rng = np.random.default_rng(0)
    y_bruite = yt_.copy()
    if bruit > 0:
        retourner = rng.random(len(y_bruite)) < bruit
        y_bruite[retourner] = 1 - y_bruite[retourner]
    m = HistGradientBoostingClassifier(random_state=0).fit(Xt_, y_bruite)
    resultats.append((int(100 * bruit), round(100 * m.score(Xv_, yv_), 2)))

table = pd.DataFrame(resultats, columns=["bruit injecté (%)", "score sur la cible PROPRE (%)"])

print("le score, en fonction de la qualité de la cible \n")
print(table.to_string(index=False))
print("\n→ le score mesure le couple (modèle, qualité de la cible) : dégradée, la cible")
print("  plafonne tout le monde — et aucun modèle ne le répare.")

In [ ]:
# 8 — l'accord entre annotateurs : de quoi parle-t-on
# La classe annote un petit lot AVANT de regarder la vérité. Remplissez les listes de la
# cellule suivante, puis on mesure le désaccord. (« nommé + démontré » : c'est l'outil qui calcule.)
lot = train.sample(20, random_state=1)

print("le lot à annoter, vingt rendez-vous tirés au hasard \n")
lot_affiche = lot[[ID] + [c for c in ("age", "delai_jours", "charge_chronique", "rappel_envoye")
                          if c in lot.columns]]
lot_affiche

In [ ]:
# 9 — vos annotations, puis le kappa de Cohen
# Exemple rempli, pour que vous voyiez à quoi ressemble un résultat. Remplacez ces deux
# listes par les 20 valeurs que la classe a VRAIMENT annotées (1 = « absent », 0 = « présent »).
# Les deux listes constantes donnent nan : un kappa n'a pas de sens quand personne ne varie.
annotateur_A = [1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0]
annotateur_B = [1,0,0,0,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0]   # trois désaccords avec A
verite = lot[CIBLE].tolist()

print("le désaccord, mesuré \n")
print("accord A-B       :", round(cohen_kappa_score(annotateur_A, annotateur_B), 3))
print("accord A-vérité  :", round(cohen_kappa_score(annotateur_A, verite), 3))

print("\nLecture : 0 = hasard · < 0,4 faible · 0,4-0,6 modéré · 0,6-0,8 bon · > 0,8 quasi parfait.")
print("Une cible à 0,7 d'accord est une cible molle : le score maximum en dépend.")

### Ce que le kappa vient de montrer

Regardez les **deux** nombres : les deux annotateurs s'accordent **modérément entre eux**
(≈ 0,6) et **pas du tout avec la vérité** (≈ 0,03).

> **L'accord entre annotateurs n'est pas l'accord avec la vérité.**

Une cible peut être consensuelle, stable, bien documentée — et fausse. C'est pourquoi on
interroge la cible **avant** de la modéliser : un modèle apprend la cible qu'on lui donne,
erreurs comprises. Aucun modèle ne répare une cible molle.

In [ ]:
# 10 — les valeurs aberrantes : erreur de saisie ou cas extrême LÉGITIME ?
# On ne supprime rien sans justification chiffrée. On regarde d'abord ce qui est possible.
print("les bornes de chaque colonne \n")
for col in ("age", "solde_credit", "montant_acte", "delai_jours", "taille_cm"):
    if col not in train.columns:
        continue
    s = train[col]
    if pd.api.types.is_numeric_dtype(s):
        print(f"  {col:<16} min {s.min():>10.2f} · max {s.max():>10.2f} · négatifs {int((s < 0).sum())}")
    else:
        print(f"  {col:<16} type texte · {s.nunique()} modalités distinctes")

print("\n→ un maximum très grand peut être un vrai cas extrême. Un minimum impossible est un bug.")
print("  La question n'est pas « est-ce loin de la moyenne ? » mais « est-ce possible ? »")

In [ ]:
# 11 — une colonne peut-elle renseigner une information connue APRÈS l'issue ?
# La fuite la plus fréquente : une colonne qui n'existerait pas au moment de décider.
# Méthode : pour chaque colonne, se demander « cette valeur est-elle connue le jour de la prise
# du rendez-vous ? ». Puis vérifier par la mesure : sa présence seule suffit-elle à prédire ?
soupcons = []
for c in train.columns:
    if c in (ID, CIBLE) or not pd.api.types.is_numeric_dtype(train[c]):
        continue
    # une colonne qui ne prédit QUE les absents, par construction, sent la fuite
    if train[c].isna().any():
        taux_manquant_absents = train.loc[train[CIBLE] == 1, c].isna().mean()
        taux_manquant_presents = train.loc[train[CIBLE] == 0, c].isna().mean()
        if abs(taux_manquant_absents - taux_manquant_presents) > 0.5:
            soupcons.append((c, round(100 * taux_manquant_absents, 1),
                             round(100 * taux_manquant_presents, 1)))

print("les colonnes suspectes, par leurs manquants \n")
if soupcons:
    print(pd.DataFrame(soupcons, columns=["colonne", "% vide chez les absents",
                                          "% vide chez les présents"]).to_string(index=False))
else:
    print("aucune colonne ne se distingue par ses manquants — cherchez par le TEMPS, pas par le vide")

## Le gabarit des 5 constats

Un constat a quatre morceaux. **Les quatre sont obligatoires** — c'est le troisième qui manque
toujours, et c'est celui qui force à réfléchir**.

In [ ]:
# 12 — le gabarit. Remplissez-le, cinq fois. Un constat sans chiffre sera rendu sans note
CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (une autre colonne, l'ensemble, un seuil)
       conséquence : ce qu'on en fait — garder, écarter, imputer, surveiller"""
    CONSTATS.append({"observation": observation, "chiffre": chiffre,
                     "comparaison": comparaison, "conséquence": consequence})
    print(pd.DataFrame(CONSTATS).to_string(index=False))

# Trois exemples — à remplacer par les vôtres :
print("un exemple de constat \n")
constat("une colonne ne sert à rien", "AUC 0,501", "1 valeur distincte sur les 80 cliniques",
        "écarter")

print("un deuxième exemple \n")
constat("deux colonnes disent la même chose", "corrélation 0,99", "contre 0,3 en moyenne ailleurs",
        "n'en garder qu'une")

print("un troisième exemple \n")
constat("la cible n'est pas équilibrée", f"{100*y.mean():.1f} % de positifs",
        "contre 50 % pour un problème équilibré", "le dire, avec le chiffre")

## Les 3 hypothèses de features

Une hypothèse n'est pas une idée : c'est une **prédiction falsifiable**. Elle dit ce qu'on
attend, comment on le testera, et **ce qui la contredirait**.

In [ ]:
# 13 — le gabarit des hypothèses
HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction falsifiable : ce qu'elle raconte, comment on la teste,
       et ce qui la contredirait."""
    HYPOTHESES.append({"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
                       "test": comment_on_la_test, "falsifiée si": ce_qui_la_contredirait})
    print(pd.DataFrame(HYPOTHESES).to_string(index=False))

# Trois exemples — à remplacer :
print("un exemple d'hypothèse \n")
hypothese("jours depuis la prise", "un délai long fait oublier le rendez-vous",
          "score avec / sans, à modèle constant", "écart < 0,2 pt")

print("une deuxième hypothèse \n")
hypothese("nombre de rendez-vous passés du patient", "un habitué manque moins",
          "score avec / sans", "écart nul sur le test")

print("une troisième hypothèse \n")
hypothese("jour de la semaine", "le lundi n'est pas le samedi",
          "taux d'absence par jour, seaux ≥ 500 lignes", "écart < 1 pt")

In [ ]:
# 14 — et ce qu'on rend : le rapport
# Le rapport n'est pas un notebook : c'est un TEXTE, avec cinq lignes de constats et trois
# hypothèses. Le voici en clair, prêt à copier dans votre dépôt.
print("CONSTATS CHIFFRÉS \n")
print(pd.DataFrame(CONSTATS).to_string(index=False))

print("HYPOTHÈSES DE FEATURES \n")
print(pd.DataFrame(HYPOTHESES).to_string(index=False))

print("Un constat sans chiffre n'est pas un constat. Une hypothèse sans falsification n'en est pas une.")